In [5]:
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go

In [6]:
from Models.EnergyStorageModel import EnergyStorageModel as ESM

In [7]:
import pandas as pd
import numpy as np

policy_train_files = {
    0.0: "./Data/policy_train.csv",
    0.15: "./Data/policy_train_all_feat_noise015.csv",
    0.25: "./Data/policy_train_all_feat_noise025.csv",
    0.35: "./Data/policy_train_all_feat_noise035.csv"
}

policy_test_files = {
    0.0: "./Data/policy_test.csv",
    0.15: "./Data/policy_test_all_feat_noise015.csv",
    0.25: "./Data/policy_test_all_feat_noise025.csv",
    0.35: "./Data/policy_test_all_feat_noise035.csv"
}

def load_and_reshape(file_path, keep_first_column=False):
    df = pd.read_csv(file_path)
    
    if not keep_first_column:
        df.drop(columns=["0"], inplace=True)
    
    array = df.to_numpy()
    num_cols = array.shape[1]
    new_length = (array.shape[0] // 24) * 24
    array = array[:new_length, :]
    reshaped_array = array.reshape(new_length // 24, 24, num_cols).transpose(2, 0, 1)
    
    return reshaped_array


In [20]:
# best blsh parameters 
theta_low = 105
theta_high = 110

theta_low_hist = 295
theta_high_hist = 300

# best badp model 
model_path = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_hist/model_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"
scaler_path = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_hist/scaler_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"

model_path_mean = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_mean/model_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"
scaler_path_mean = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_mean/scaler_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"

model_path_comb = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_comb/model_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"
scaler_path_comb = "/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_comb/scaler_sample_size_60_discount_factor_0_99_energy_bonus_factor_0_09.pkl"

In [17]:
test_scenarios = load_and_reshape(policy_test_files[0.0], keep_first_column=True)[0:11, :, :]

In [29]:
from Models.Policies.PFA import BuyLowSellHigh as BLSH
from Models.Policies.VFA import BADP

all_results = {}
for scenario in range(test_scenarios.shape[0]):
    data = test_scenarios[scenario, :, :]

    test_model = ESM(
        seed=0,
        t0=0,
        T=len(data[1:]),
        S0={"energy_amount": 300, "price": data[0]},
        init_args={"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50},
        exog_params={"hist_price": data[1:]},
        model_name="cnf-24"
    )

    blsh_policy = BLSH(
        model=test_model,
        policy_name="test_blsh_multiple_scenarios",
        theta_low=theta_low,
        theta_high=theta_high,
        verbose=False
    )

    blsh_policy_hist = BLSH(
        model=test_model,
        policy_name="test_blsh_multiple_scenarios",
        theta_low=theta_low_hist,
        theta_high=theta_high_hist,
        verbose=False
    )

    blsh_result = blsh_policy.run_policy()
    blsh_result_hist = blsh_policy_hist.run_policy()

    all_results[scenario] = {
        "blsh": blsh_result,
        "blsh_hist": blsh_result_hist   
    }

In [30]:
all_results_df = pd.DataFrame(all_results).T

In [50]:
all_results_df.mean(axis=0)

blsh         6207.743510
blsh_hist     304.060975
dtype: float64

In [39]:
from Models.Policies.PFA import BuyLowSellHigh as BLSH
from Models.Policies.VFA import BADP

all_results_badp = {}
for scenario in range(test_scenarios.shape[0]):

    data = test_scenarios[scenario, :, :]

    test_model = ESM(
        seed=0,
        t0=0,
        T=len(data[1:]),
        S0={"energy_amount": 300, "price": data[0]},
        init_args={"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50},
        exog_params={"hist_price": data[1:]},
        model_name="cnf-24"
    )

    badp_policy = BADP(
        model=test_model,
        price_samples=data,
        sample_size=24,
        discount_factor=0.99,
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=0.09
    )
    badp_policy.load_model(model_path, scaler_path) 

    badp_policy_mean = BADP(
        model=test_model,
        price_samples=data,
        sample_size=24,
        discount_factor=0.99,
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=0.09
    )
    badp_policy_mean.load_model(model_path_mean, scaler_path_mean) 

    badp_policy_comb = BADP(
        model=test_model,
        price_samples=data,
        sample_size=24,
        discount_factor=0.99,
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=0.09
    )
    badp_policy_comb.load_model(model_path_comb, scaler_path_comb) 

    badp_resul_hist = badp_policy.run_policy()
    badp_result_mean = badp_policy_mean.run_policy()
    badp_result_comb = badp_policy_comb.run_policy()

    all_results_badp[scenario] = {
        "badp_hist": badp_resul_hist,
        "badp_mean": badp_result_mean,
        "badp_comb": badp_result_comb
    }

all_results_df_badp = pd.DataFrame(all_results_badp).T

In [45]:
all_results_df_badp

,badp_hist,badp_mean,badp_comb
0,8104.743961,8110.050538,8108.249838
1,10640.421395,10635.667050,10633.509260
2,10380.794354,10411.358869,10413.793334
3,10415.587754,10471.744131,10472.070088
4,10342.282424,10351.932396,10349.708443
5,10474.909404,10489.581329,10496.181413
6,10386.937190,10385.980832,10385.692708
7,10511.040676,10527.323907,10527.479067
8,10484.161776,10501.832883,10500.994168
9,10519.491656,10559.764503,10557.863658


In [42]:
all_results_df_badp.mean(axis=0)

badp_hist    10248.059214
badp_mean    10267.457719
badp_comb    10267.615995
dtype: float64

In [48]:
8110.05 / 8104.74

1.0006551721585146